
# RQ29A — Production model refit, serialization, and release

## Scientific / production question

**Which fitted artifacts instantiate the architecture frozen in RQ28, and can those artifacts be serialized and reloaded without changing their predictions?**

RQ29A is deliberately separated from model characterization (RQ29B) and end-to-end inference/CLI engineering (RQ29C).

### Frozen contract

- Consume the **RQ28 production architecture** exactly as frozen.
- Production endpoints are `global_mixed`, `multiclass_clean`, and the three subtype auxiliaries.
- `global_easy` and `global_hard` remain benchmark-only.
- All retained production members use `esm2_t30_150m`.
- Final production estimators are **refit on the complete representation-eligible Stage-06 task dataset**.
- A production model is **not** chosen from one `R/S` benchmark cell. `R` and `S` were evaluation perturbations, not deployment hyperparameters.
- Final estimator configuration is consolidated from the 16 benchmark cells **using validation-only evidence**:
  1. take the modal validation-selected `config_id`;
  2. if tied, use median validation MCC;
  3. then the frozen validation secondary metric (AP for binary; macro-F1 for multiclass);
  4. then `config_order`;
  5. then `config_id`.
- Outer-test performance is never used to choose the production configuration.
- PLM preprocessing follows the frozen benchmark contract:
  - `StandardScaler` for LR, KNN, and MLP;
  - raw FP32 embeddings for XGBoost.
- Ensembles are represented explicitly by `ensemble.json`; member estimators are serialized independently.
- Every fitted member must pass a round-trip invariant:
  `predict -> serialize -> reload -> predict`, with identical class predictions and probabilities within numerical tolerance.

### Output layout

```text
models/phagelysclass_v1/
├── manifest.json
├── checksums.tsv
├── representation/
│   └── esm2_t30_150m.json
└── tasks/
    ├── global_mixed/
    ├── multiclass_clean/
    ├── subtype_endolysin/
    ├── subtype_depolymerase/
    └── subtype_val/
        ├── members/*.joblib
        ├── members/*.metadata.json
        ├── ensemble.json          # ensemble recipes only
        └── metadata.json
```

RQ29B will add held-out characterization and `performance.json`; RQ29A does **not** evaluate the all-data refits on their fitting data.


In [1]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import inspect
import json
import os
import platform
import shutil
import time
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 300)
pd.set_option("display.max_rows", 300)

SEED = 42
N_JOBS = -1
PROBE_N = 8
ROUNDTRIP_ATOL = 1e-10
ALLOW_REBUILD = False

MODEL_VERSION = "phagelysclass_v1"
REPRESENTATION_ID = "esm2_t30_150m"
REPRESENTATION_CHECKPOINT = "facebook/esm2_t30_150m_ur50d"
FEATURE_COLS = [f"p_{i}" for i in range(640)]
CANONICAL_AA = frozenset("ACDEFGHIKLMNPQRSTVWY")
R_LEVELS = [90, 70, 50, 30]
S_LEVELS = [90, 70, 50, 30]

PRODUCTION_TASKS = [
    "global_mixed",
    "multiclass_clean",
    "subtype_endolysin",
    "subtype_depolymerase",
    "subtype_val",
]

TASK_SPECS = {
    "global_mixed": {
        "path": "global_mixed",
        "type": "binary",
        "component_id": "core_global_lytic",
    },
    "multiclass_clean": {
        "path": "multiclass_clean",
        "type": "multiclass",
        "component_id": "core_function_multiclass",
    },
    "subtype_endolysin": {
        "path": "subtype_binary/endolysin",
        "type": "binary",
        "component_id": "aux_endolysin_compatibility",
    },
    "subtype_depolymerase": {
        "path": "subtype_binary/depolymerase",
        "type": "binary",
        "component_id": "aux_depolymerase_compatibility",
    },
    "subtype_val": {
        "path": "subtype_binary/virion_associated_lytic_enzyme",
        "type": "binary",
        "component_id": "aux_VAL_compatibility",
    },
}

MULTICLASS_ORDER = [
    "endolysin",
    "depolymerase",
    "virion_associated_lytic_enzyme",
]


def find_project_root(start=None):
    explicit = os.environ.get("PHAGELYSCLASS_ROOT")
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser().resolve())
    start = Path.cwd() if start is None else Path(start)
    candidates.extend([start, *start.parents])

    seen = set()
    for root in candidates:
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)
        if (
            (root / "results/analysis_deployment/rq28/rq28_frozen_components.tsv").is_file()
            and (root / "results/analysis_deployment/rq28/rq28_production_architecture.json").is_file()
            and (root / "results/analysis_supervised/stage12a/supervised_master_results.tsv").is_file()
            and (root / "results/training_supervised").is_dir()
            and (root / "data/tasks").is_dir()
            and (root / "config/representation_registry/representation_registry.tsv").is_file()
        ):
            return root

    raise FileNotFoundError(
        "Could not locate PhageLysClass root with RQ28, Stage12A, Stage06 tasks, "
        "training_supervised, and representation registry."
    )


ROOT = find_project_root()
RQ28_DIR = ROOT / "results/analysis_deployment/rq28"
STAGE12A = ROOT / "results/analysis_supervised/stage12a"
STAGE12A_MASTER_PATH = STAGE12A / "supervised_master_results.tsv"
TRAINING_ROOT = ROOT / "results/training_supervised"
TASK_ROOT = ROOT / "data/tasks"
REGISTRY_PATH = ROOT / "config/representation_registry/representation_registry.tsv"

RQ29A_OUT = ROOT / "results/analysis_deployment/rq29a"
RQ29A_OUT.mkdir(parents=True, exist_ok=True)

MODELS_PARENT = ROOT / "models"
FINAL_MODEL_ROOT = MODELS_PARENT / MODEL_VERSION
STAGING_ROOT = MODELS_PARENT / f".{MODEL_VERSION}.rq29a_staging"
MODELS_PARENT.mkdir(parents=True, exist_ok=True)

if FINAL_MODEL_ROOT.exists() and not ALLOW_REBUILD:
    raise FileExistsError(
        f"Production release already exists: {FINAL_MODEL_ROOT}\n"
        "Set ALLOW_REBUILD=True only if you intentionally want to replace it after "
        "a complete successful rebuild."
    )

if STAGING_ROOT.exists():
    shutil.rmtree(STAGING_ROOT)
STAGING_ROOT.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("RQ29A_OUT:", RQ29A_OUT)
print("STAGING_ROOT:", STAGING_ROOT)
print("FINAL_MODEL_ROOT:", FINAL_MODEL_ROOT)
print("Production refit uses full Stage-06 task evidence; no R/S cell is treated as the final training set.")


ROOT: /home/david/Desktop/colabs/inta_project/phagelytic_core_project/phagelysclass
RQ29A_OUT: /home/david/Desktop/colabs/inta_project/phagelytic_core_project/phagelysclass/results/analysis_deployment/rq29a
STAGING_ROOT: /home/david/Desktop/colabs/inta_project/phagelytic_core_project/phagelysclass/models/.phagelysclass_v1.rq29a_staging
FINAL_MODEL_ROOT: /home/david/Desktop/colabs/inta_project/phagelytic_core_project/phagelysclass/models/phagelysclass_v1
Production refit uses full Stage-06 task evidence; no R/S cell is treated as the final training set.


## 1. Freeze and audit the RQ28 architecture

In [2]:

RQ28_COMPONENTS_PATH = RQ28_DIR / "rq28_frozen_components.tsv"
RQ28_ARCH_PATH = RQ28_DIR / "rq28_production_architecture.json"

frozen = pd.read_csv(RQ28_COMPONENTS_PATH, sep="\t", low_memory=False)
architecture = json.loads(RQ28_ARCH_PATH.read_text(encoding="utf-8"))


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n", ""}:
        return False
    raise ValueError(f"Cannot parse boolean value: {value!r}")


master = pd.read_csv(STAGE12A_MASTER_PATH, sep="\t", low_memory=False)
if len(master) != 2240:
    raise RuntimeError(
        f"Stage12A supervised master must contain 2,240 experiment families; "
        f"observed {len(master)}."
    )

required_cols = {
    "task", "component_id", "tier", "default_execution",
    "option_type", "voting_scheme", "n_members",
    "member_strategy_ids", "member_representations", "member_algorithms",
}
missing = required_cols - set(frozen.columns)
if missing:
    raise KeyError(f"RQ28 frozen components missing columns: {sorted(missing)}")

if architecture.get("architecture_id") != "C_2_core_plus_3_aux":
    raise RuntimeError(
        f"Unexpected RQ28 architecture_id: {architecture.get('architecture_id')}"
    )
if architecture.get("production_global_task") != "global_mixed":
    raise RuntimeError(
        f"Unexpected production global task: {architecture.get('production_global_task')}"
    )

frozen = (
    frozen[frozen["task"].isin(PRODUCTION_TASKS)]
    .copy()
    .set_index("task")
    .reindex(PRODUCTION_TASKS)
    .reset_index()
)

if len(frozen) != 5 or frozen["task"].nunique() != 5:
    raise RuntimeError("RQ28 must define exactly the five frozen production components.")


def split_semicolon(value):
    return [x.strip() for x in str(value).split(";") if x.strip()]


member_rows = []
for row in frozen.to_dict("records"):
    strategies = split_semicolon(row["member_strategy_ids"])
    reps = split_semicolon(row["member_representations"])
    algs = split_semicolon(row["member_algorithms"])
    n_members = int(row["n_members"])

    if not (len(strategies) == len(reps) == len(algs) == n_members):
        raise RuntimeError(
            f"RQ28 member-vector mismatch for {row['task']}: "
            f"n_members={n_members}, strategies={strategies}, reps={reps}, algs={algs}"
        )

    for i, (strategy_id, rep, alg) in enumerate(zip(strategies, reps, algs), start=1):
        member_rows.append({
            "task": row["task"],
            "component_id": row["component_id"],
            "tier": row["tier"],
            "default_execution": as_bool(row["default_execution"]),
            "option_type": row["option_type"],
            "voting_scheme": row["voting_scheme"],
            "member_index": i,
            "strategy_id": strategy_id,
            "representation": rep,
            "algorithm": alg,
        })

members = pd.DataFrame(member_rows)

if set(members["representation"]) != {REPRESENTATION_ID}:
    raise RuntimeError(
        "RQ29A expects the RQ28 production freeze to converge entirely to ESM2-T30. "
        f"Observed: {sorted(members['representation'].unique())}"
    )
if members.duplicated(["task", "algorithm"]).any():
    raise RuntimeError("A task contains duplicate production members with the same algorithm.")
if len(members) != 10:
    raise RuntimeError(
        f"Frozen RQ28 architecture currently implies {len(members)} fitted members; expected 10."
    )
if (frozen["voting_scheme"] == "validation_weighted_soft").any():
    raise RuntimeError("Validation-weighted ensemble entered RQ28 production recipe unexpectedly.")

display(frozen[[
    "task", "component_id", "tier", "default_execution",
    "option_type", "voting_scheme", "n_members",
    "member_strategy_ids",
]])
display(members)

print("RQ28_ARCHITECTURE_PREFLIGHT_OK")


,task,component_id,tier,default_execution,option_type,voting_scheme,n_members,member_strategy_ids
0,global_mixed,core_global_lytic,core,True,ensemble,unweighted_soft,2,esm2_t30_150m + KNN;esm2_t30_150m + MLP
1,multiclass_clean,core_function_multiclass,core,True,ensemble,unweighted_soft,2,esm2_t30_150m + XGB;esm2_t30_150m + LR
2,subtype_endolysin,aux_endolysin_compatibility,auxiliary,False,ensemble,unweighted_soft,3,esm2_t30_150m + KNN;esm2_t30_150m + XGB;esm2_t...
3,subtype_depolymerase,aux_depolymerase_compatibility,auxiliary,False,ensemble,unweighted_soft,2,esm2_t30_150m + MLP;esm2_t30_150m + KNN
4,subtype_val,aux_VAL_compatibility,auxiliary,False,single,none,1,esm2_t30_150m + MLP


,task,component_id,tier,default_execution,option_type,voting_scheme,member_index,strategy_id,representation,algorithm
0,global_mixed,core_global_lytic,core,True,ensemble,unweighted_soft,1,esm2_t30_150m + KNN,esm2_t30_150m,KNN
1,global_mixed,core_global_lytic,core,True,ensemble,unweighted_soft,2,esm2_t30_150m + MLP,esm2_t30_150m,MLP
2,multiclass_clean,core_function_multiclass,core,True,ensemble,unweighted_soft,1,esm2_t30_150m + XGB,esm2_t30_150m,XGB
3,multiclass_clean,core_function_multiclass,core,True,ensemble,unweighted_soft,2,esm2_t30_150m + LR,esm2_t30_150m,LR
4,subtype_endolysin,aux_endolysin_compatibility,auxiliary,False,ensemble,unweighted_soft,1,esm2_t30_150m + KNN,esm2_t30_150m,KNN
5,subtype_endolysin,aux_endolysin_compatibility,auxiliary,False,ensemble,unweighted_soft,2,esm2_t30_150m + XGB,esm2_t30_150m,XGB
6,subtype_endolysin,aux_endolysin_compatibility,auxiliary,False,ensemble,unweighted_soft,3,esm2_t30_150m + MLP,esm2_t30_150m,MLP
7,subtype_depolymerase,aux_depolymerase_compatibility,auxiliary,False,ensemble,unweighted_soft,1,esm2_t30_150m + MLP,esm2_t30_150m,MLP
8,subtype_depolymerase,aux_depolymerase_compatibility,auxiliary,False,ensemble,unweighted_soft,2,esm2_t30_150m + KNN,esm2_t30_150m,KNN
9,subtype_val,aux_VAL_compatibility,auxiliary,False,single,none,1,esm2_t30_150m + MLP,esm2_t30_150m,MLP


RQ28_ARCHITECTURE_PREFLIGHT_OK



## 2. Consolidate one final estimator configuration per frozen member

The benchmark selected a configuration independently inside every `task × R × S × representation × algorithm` family using validation data only. RQ29A now needs one production configuration for each retained member recipe without declaring one R/S cell to be “the production split”.

The consolidation below is therefore a **validation-consensus operation across the 16 R/S cells**. It explicitly re-checks the frozen validation selection rule in every cell and does not read `outer_test_metrics.tsv`.


In [3]:

def experiment_dir(task, r, s, representation, algorithm):
    return (
        TRAINING_ROOT
        / TASK_SPECS[task]["path"]
        / f"R{r}_S{s}"
        / representation
        / algorithm
    )


def recompute_selected_config(validation, task_type):
    v = validation.copy()
    for col in ["mcc", "config_order"]:
        if col not in v.columns:
            raise KeyError(f"Validation table lacks {col}")
        v[col] = pd.to_numeric(v[col], errors="raise")

    if task_type == "binary":
        if "ap" not in v.columns:
            raise KeyError("Binary validation table lacks AP.")
        v["ap"] = pd.to_numeric(v["ap"], errors="raise")
        ranked = v.sort_values(
            ["mcc", "ap", "config_order", "config_id"],
            ascending=[False, False, True, True],
            kind="mergesort",
        )
    else:
        if "macro_f1" not in v.columns:
            raise KeyError("Multiclass validation table lacks macro_f1.")
        v["macro_f1"] = pd.to_numeric(v["macro_f1"], errors="raise")
        ranked = v.sort_values(
            ["mcc", "macro_f1", "config_order", "config_id"],
            ascending=[False, False, True, True],
            kind="mergesort",
        )

    return str(ranked.iloc[0]["config_id"])


selected_cell_rows = []
validation_rows = []

for rec in members.to_dict("records"):
    task = rec["task"]
    task_type = TASK_SPECS[task]["type"]

    for r in R_LEVELS:
        for s in S_LEVELS:
            exp_dir = experiment_dir(
                task, r, s, rec["representation"], rec["algorithm"]
            )
            validation_path = exp_dir / "validation_metrics.tsv"
            selected_path = exp_dir / "selected_configuration.json"

            if not validation_path.is_file() or not selected_path.is_file():
                raise FileNotFoundError(
                    f"Missing validation-selection artifact for production member: {exp_dir}"
                )

            validation = pd.read_csv(validation_path, sep="\t", low_memory=False)
            selected_json = json.loads(selected_path.read_text(encoding="utf-8"))
            selected_id = str(selected_json.get("config_id", "")).strip()
            recomputed_id = recompute_selected_config(validation, task_type)

            if not selected_id:
                raise RuntimeError(f"Empty selected config in {selected_path}")
            if selected_id != recomputed_id:
                raise RuntimeError(
                    f"Frozen validation selection mismatch at {task} R{r}/S{s} "
                    f"{rec['algorithm']}: stored={selected_id}, recomputed={recomputed_id}"
                )

            selected_cell_rows.append({
                "task": task,
                "representation": rec["representation"],
                "algorithm": rec["algorithm"],
                "strategy_id": rec["strategy_id"],
                "R": r,
                "S": s,
                "selected_config_id": selected_id,
                "recomputed_config_id": recomputed_id,
                "selection_match": True,
            })

            vv = validation.copy()
            vv["task"] = task
            vv["task_type"] = task_type
            vv["representation"] = rec["representation"]
            vv["algorithm"] = rec["algorithm"]
            vv["strategy_id"] = rec["strategy_id"]
            vv["R"] = r
            vv["S"] = s
            validation_rows.append(vv)

selected_cells = pd.DataFrame(selected_cell_rows)
validation_all = pd.concat(validation_rows, ignore_index=True)

if len(selected_cells) != len(members) * 16:
    raise RuntimeError(
        f"Expected {len(members) * 16} selected member-cells; got {len(selected_cells)}"
    )
if not selected_cells["selection_match"].all():
    raise RuntimeError("At least one stored validation selection does not reproduce.")


def consolidate_member_config(cell_group, val_group, task_type):
    counts = (
        cell_group["selected_config_id"]
        .value_counts()
        .rename_axis("config_id")
        .reset_index(name="selected_count")
    )
    max_count = int(counts["selected_count"].max())
    tied = counts.loc[counts["selected_count"].eq(max_count), "config_id"].astype(str).tolist()

    score_rows = []
    for config_id in tied:
        g = val_group[val_group["config_id"].astype(str).eq(config_id)].copy()
        if len(g) == 0:
            raise RuntimeError(f"No validation rows found for tied config {config_id}")

        row = {
            "config_id": config_id,
            "selected_count": int(
                counts.loc[counts["config_id"].astype(str).eq(config_id), "selected_count"].iloc[0]
            ),
            "median_validation_mcc": float(pd.to_numeric(g["mcc"], errors="raise").median()),
            "median_config_order": float(pd.to_numeric(g["config_order"], errors="raise").median()),
        }
        if task_type == "binary":
            row["median_validation_secondary"] = float(
                pd.to_numeric(g["ap"], errors="raise").median()
            )
        else:
            row["median_validation_secondary"] = float(
                pd.to_numeric(g["macro_f1"], errors="raise").median()
            )
        score_rows.append(row)

    score = pd.DataFrame(score_rows).sort_values(
        [
            "selected_count",
            "median_validation_mcc",
            "median_validation_secondary",
            "median_config_order",
            "config_id",
        ],
        ascending=[False, False, False, True, True],
        kind="mergesort",
    ).reset_index(drop=True)

    winner = score.iloc[0]
    return {
        "consensus_config_id": str(winner["config_id"]),
        "modal_selected_count": int(winner["selected_count"]),
        "selection_fraction": float(winner["selected_count"] / 16.0),
        "n_distinct_selected_configs": int(cell_group["selected_config_id"].nunique()),
        "modal_tie_size": int(len(tied)),
        "tie_break_used": bool(len(tied) > 1),
        "consensus_median_validation_mcc": float(winner["median_validation_mcc"]),
        "consensus_median_validation_secondary": float(
            winner["median_validation_secondary"]
        ),
        "consensus_median_config_order": float(winner["median_config_order"]),
        "selected_config_counts": ";".join(
            f"{r.config_id}:{int(r.selected_count)}"
            for r in counts.sort_values(["selected_count", "config_id"], ascending=[False, True]).itertuples()
        ),
    }


consensus_rows = []
for rec in members.to_dict("records"):
    mask = (
        selected_cells["task"].eq(rec["task"])
        & selected_cells["representation"].eq(rec["representation"])
        & selected_cells["algorithm"].eq(rec["algorithm"])
    )
    cell_group = selected_cells[mask].copy()

    vm = (
        validation_all["task"].eq(rec["task"])
        & validation_all["representation"].eq(rec["representation"])
        & validation_all["algorithm"].eq(rec["algorithm"])
    )
    val_group = validation_all[vm].copy()

    if len(cell_group) != 16:
        raise RuntimeError(f"Incomplete R/S config-selection coverage for {rec}")

    consensus = consolidate_member_config(
        cell_group, val_group, TASK_SPECS[rec["task"]]["type"]
    )
    consensus_rows.append({**rec, **consensus})

config_consensus = pd.DataFrame(consensus_rows)

selected_cells.to_csv(
    RQ29A_OUT / "rq29a_selected_config_cells.tsv", sep="\t", index=False
)
config_consensus.to_csv(
    RQ29A_OUT / "rq29a_config_consensus.tsv", sep="\t", index=False
)

display(config_consensus[[
    "task", "algorithm", "consensus_config_id",
    "modal_selected_count", "selection_fraction",
    "n_distinct_selected_configs", "tie_break_used",
    "selected_config_counts",
]])

print("VALIDATION_ONLY_CONFIG_CONSENSUS_OK")


,task,algorithm,consensus_config_id,modal_selected_count,selection_fraction,n_distinct_selected_configs,tie_break_used,selected_config_counts
0,global_mixed,KNN,KNN02,8,0.5000,3,False,KNN02:8;KNN01:5;KNN03:3
1,global_mixed,MLP,MLP02,8,0.5000,3,False,MLP02:8;MLP03:6;MLP01:2
2,multiclass_clean,XGB,XGB01,8,0.5000,3,False,XGB01:8;XGB03:6;XGB02:2
3,multiclass_clean,LR,LR01,10,0.6250,3,False,LR01:10;LR03:4;LR02:2
4,subtype_endolysin,KNN,KNN01,8,0.5000,3,False,KNN01:8;KNN02:4;KNN03:4
5,subtype_endolysin,XGB,XGB01,7,0.4375,3,True,XGB01:7;XGB03:7;XGB02:2
6,subtype_endolysin,MLP,MLP02,9,0.5625,3,False,MLP02:9;MLP03:5;MLP01:2
7,subtype_depolymerase,MLP,MLP01,9,0.5625,3,False,MLP01:9;MLP02:4;MLP03:3
8,subtype_depolymerase,KNN,KNN03,6,0.3750,3,False,KNN03:6;KNN01:5;KNN02:5
9,subtype_val,MLP,MLP01,6,0.3750,3,True,MLP01:6;MLP02:6;MLP03:4


VALIDATION_ONLY_CONFIG_CONSENSUS_OK


## 3. Frozen parameter panels and production estimator factory

In [4]:

CONFIGS = {
    "LR01": {"algorithm": "LR", "C": 0.1},
    "LR02": {"algorithm": "LR", "C": 1.0},
    "LR03": {"algorithm": "LR", "C": 10.0},

    "KNN01": {"algorithm": "KNN", "n_neighbors": 3},
    "KNN02": {"algorithm": "KNN", "n_neighbors": 5},
    "KNN03": {"algorithm": "KNN", "n_neighbors": 11},

    "XGB01": {
        "algorithm": "XGB", "n_estimators": 300, "max_depth": 4,
        "learning_rate": 0.10,
    },
    "XGB02": {
        "algorithm": "XGB", "n_estimators": 500, "max_depth": 6,
        "learning_rate": 0.05,
    },
    "XGB03": {
        "algorithm": "XGB", "n_estimators": 800, "max_depth": 4,
        "learning_rate": 0.03,
    },

    "MLP01": {
        "algorithm": "MLP", "hidden_layer_sizes": (256,),
        "alpha": 1e-4, "learning_rate_init": 1e-3,
    },
    "MLP02": {
        "algorithm": "MLP", "hidden_layer_sizes": (512, 128),
        "alpha": 1e-4, "learning_rate_init": 5e-4,
    },
    "MLP03": {
        "algorithm": "MLP", "hidden_layer_sizes": (256, 128, 64),
        "alpha": 1e-3, "learning_rate_init": 5e-4,
    },
}

for row in config_consensus.itertuples(index=False):
    config_id = row.consensus_config_id
    if config_id not in CONFIGS:
        raise KeyError(f"Consensus config {config_id} is absent from frozen parameter panel.")
    if CONFIGS[config_id]["algorithm"] != row.algorithm:
        raise RuntimeError(
            f"Config/algorithm mismatch: {config_id} vs {row.algorithm}"
        )

print("FROZEN_PARAMETER_PANEL_OK")


FROZEN_PARAMETER_PANEL_OK


## 4. Resolve and validate the shared ESM2-T30 representation artifact

In [5]:

registry = pd.read_csv(REGISTRY_PATH, sep="\t", dtype=str).fillna("")
if not {"representation_id", "status"}.issubset(registry.columns):
    raise KeyError(
        "Representation registry must contain representation_id and status."
    )

rep_row = registry[registry["representation_id"].eq(REPRESENTATION_ID)].copy()
if len(rep_row) != 1:
    raise RuntimeError(
        f"Expected exactly one registry row for {REPRESENTATION_ID}; observed {len(rep_row)}"
    )
rep_row = rep_row.iloc[0]

if str(rep_row["status"]).upper() != "READY":
    raise RuntimeError(
        f"{REPRESENTATION_ID} is not READY in representation registry."
    )

if "dimension" in registry.columns and str(rep_row.get("dimension", "")).strip():
    observed_dimension = int(float(str(rep_row["dimension"]).strip()))
    if observed_dimension != 640:
        raise RuntimeError(
            f"{REPRESENTATION_ID}: registry dimension={observed_dimension}, expected=640."
        )


FROZEN_T30_FALLBACK = (
    ROOT
    / "data/representations/sylphy_embeddings/models"
    / "facebook_esm2_t30_150m_ur50d"
    / "embeddings_last_mean_fp32.parquet"
)


def resolve_representation_path(row):
    path_columns = [
        "file_path", "artifact_path", "representation_path",
        "matrix_path", "output_path", "path",
    ]
    candidates = []
    for col in path_columns:
        if col in row.index and str(row[col]).strip():
            p = Path(str(row[col]).strip()).expanduser()
            if not p.is_absolute():
                p = ROOT / p
            candidates.append(p.resolve())

    existing = [p for p in candidates if p.is_file()]
    if len(existing) == 1:
        return existing[0]
    if len(existing) > 1:
        uniq = sorted(set(existing))
        if len(uniq) == 1:
            return uniq[0]
        raise RuntimeError(
            "Representation registry points to multiple existing artifacts:\n"
            + "\n".join(map(str, uniq))
        )

    # Exact frozen Stage10D fallback used by the benchmark/candidate pipeline.
    if FROZEN_T30_FALLBACK.is_file():
        return FROZEN_T30_FALLBACK.resolve()

    # Conservative recursive fallback only if registry and exact frozen path fail.
    rep_root = ROOT / "data/representations"
    hits = sorted({
        p.resolve()
        for p in rep_root.rglob("*.parquet")
        if REPRESENTATION_ID.lower() in str(p).lower()
    })
    if len(hits) == 1:
        return hits[0]

    raise FileNotFoundError(
        f"Could not uniquely resolve the READY {REPRESENTATION_ID} parquet artifact. "
        f"Registry candidates={list(map(str, candidates))}; recursive hits={list(map(str, hits[:20]))}"
    )


REP_PATH = resolve_representation_path(rep_row)
if REP_PATH.suffix.lower() != ".parquet":
    raise RuntimeError(
        f"RQ29A currently expects the frozen T30 representation as parquet: {REP_PATH}"
    )

print("Representation:", REPRESENTATION_ID)
print("Artifact:", REP_PATH)
print("Registry status:", rep_row["status"])
print("REPRESENTATION_PATH_RESOLVED")


Representation: esm2_t30_150m
Artifact: /home/david/Desktop/colabs/inta_project/phagelytic_core_project/phagelysclass/data/representations/sylphy_embeddings/models/facebook_esm2_t30_150m_ur50d/embeddings_last_mean_fp32.parquet
Registry status: READY
REPRESENTATION_PATH_RESOLVED



## 5. Build the full Stage-06 production evidence tables

Only the common frozen PLM eligibility rule is applied here:

- canonical 20-aa alphabet;
- sequence length `<=1024`.

No H90/H70/H50/H30 redundancy reduction is imposed on the final all-data refit. Exact duplicate sequences must have a consistent task label and are collapsed by `sequence_uid`.


In [6]:

def normalize_sequence(series):
    return (
        series.astype(str)
        .str.replace(r"\s+", "", regex=True)
        .str.upper()
        .str.strip()
    )


def sequence_uid(seq):
    return hashlib.sha256(seq.encode("ascii")).hexdigest()


def binary_labels(series, context):
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().all():
        vals = set(numeric.astype(int).unique())
        if vals.issubset({0, 1}) and vals == {0, 1}:
            return numeric.astype(int)

    s = series.astype(str).str.strip().str.lower()
    mapping = {
        "0": 0, "negative": 0, "neg": 0, "false": 0, "no": 0,
        "1": 1, "positive": 1, "pos": 1, "true": 1, "yes": 1,
    }
    mapped = s.map(mapping)
    if mapped.notna().all() and set(mapped.astype(int).unique()) == {0, 1}:
        return mapped.astype(int)

    raise ValueError(
        f"{context}: cannot normalize binary label column to {{0,1}}. "
        f"Observed values: {sorted(series.astype(str).unique())[:20]}"
    )


def canonical_multiclass_name(value):
    s = str(value).strip().lower()
    normalized = (
        s.replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )
    if "endolysin" in normalized:
        return "endolysin"
    if "depolymerase" in normalized:
        return "depolymerase"
    if (
        normalized in {"val", "virion_associated_lytic_enzyme"}
        or "virion_associated_lytic" in normalized
    ):
        return "virion_associated_lytic_enzyme"
    return None


task_frames = {}
task_audit_rows = []
label_mapping_rows = []
union_eligible_uids = set()

for task in PRODUCTION_TASKS:
    spec = TASK_SPECS[task]
    path = TASK_ROOT / spec["path"] / "dataset.tsv.gz"
    if not path.is_file():
        raise FileNotFoundError(path)

    df = pd.read_csv(path, sep="\t", compression="infer", low_memory=False)
    required = {"sequence", "label"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"{task}: Stage-06 dataset missing {sorted(missing)}")

    df["sequence"] = normalize_sequence(df["sequence"])
    if (df["sequence"] == "").any():
        raise ValueError(f"{task}: empty sequences found.")

    df["sequence_uid"] = df["sequence"].map(sequence_uid)
    df["length"] = df["sequence"].str.len()
    df["is_canonical_20aa"] = df["sequence"].map(
        lambda x: set(x).issubset(CANONICAL_AA)
    )
    df["eligible_t30"] = df["is_canonical_20aa"] & df["length"].le(1024)

    if spec["type"] == "binary":
        df["y"] = binary_labels(df["label"], task)
        for raw_label, y in (
            df[["label", "y"]]
            .astype({"label": str})
            .drop_duplicates()
            .itertuples(index=False, name=None)
        ):
            label_mapping_rows.append({
                "task": task,
                "raw_label": str(raw_label),
                "production_label": int(y),
                "canonical_name": "positive" if int(y) == 1 else "negative",
            })
    else:
        if "canonical_target_class" not in df.columns:
            raise KeyError(
                f"{task}: canonical_target_class is required to freeze explicit production class order."
            )

        df["canonical_class"] = df["canonical_target_class"].map(
            canonical_multiclass_name
        )
        if df["canonical_class"].isna().any():
            bad = sorted(
                df.loc[df["canonical_class"].isna(), "canonical_target_class"]
                .astype(str).unique()
            )
            raise ValueError(
                f"{task}: unmapped canonical_target_class values: {bad[:20]}"
            )

        if set(df["canonical_class"].unique()) != set(MULTICLASS_ORDER):
            raise RuntimeError(
                f"{task}: expected exactly {MULTICLASS_ORDER}; "
                f"observed {sorted(df['canonical_class'].unique())}"
            )

        # Validate that the frozen Stage-06 label is one-to-one with biological class.
        relation = df[["label", "canonical_class"]].drop_duplicates()
        if relation.groupby("label")["canonical_class"].nunique().max() != 1:
            raise RuntimeError(
                f"{task}: one Stage-06 label maps to multiple biological classes."
            )
        if relation.groupby("canonical_class")["label"].nunique().max() != 1:
            raise RuntimeError(
                f"{task}: one biological class maps to multiple Stage-06 labels."
            )

        class_to_int = {name: i for i, name in enumerate(MULTICLASS_ORDER)}
        df["y"] = df["canonical_class"].map(class_to_int).astype(int)

        for row in relation.itertuples(index=False):
            label_mapping_rows.append({
                "task": task,
                "raw_label": str(row.label),
                "production_label": int(class_to_int[row.canonical_class]),
                "canonical_name": row.canonical_class,
            })

    # Exact duplicates are allowed only if their production labels agree.
    conflict = df.groupby("sequence_uid")["y"].nunique()
    if (conflict > 1).any():
        raise RuntimeError(
            f"{task}: exact sequence duplicates carry conflicting labels for "
            f"{int((conflict > 1).sum())} sequence_uids."
        )

    n_raw = len(df)
    n_exact_unique = int(df["sequence_uid"].nunique())
    eligible = (
        df.loc[df["eligible_t30"], ["sequence_uid", "sequence", "y"]]
        .drop_duplicates("sequence_uid")
        .sort_values("sequence_uid")
        .reset_index(drop=True)
    )

    if spec["type"] == "binary" and set(eligible["y"].unique()) != {0, 1}:
        raise RuntimeError(f"{task}: both binary classes must survive T30 eligibility.")
    if spec["type"] == "multiclass" and set(eligible["y"].unique()) != {0, 1, 2}:
        raise RuntimeError(f"{task}: all three classes must survive T30 eligibility.")

    task_frames[task] = eligible
    union_eligible_uids.update(eligible["sequence_uid"])

    counts = eligible["y"].value_counts().sort_index().to_dict()
    task_audit_rows.append({
        "task": task,
        "task_type": spec["type"],
        "stage06_path": str(path),
        "n_stage06_rows": n_raw,
        "n_stage06_exact_unique": n_exact_unique,
        "n_noncanonical_rows": int((~df["is_canonical_20aa"]).sum()),
        "n_length_gt_1024_rows": int(df["length"].gt(1024).sum()),
        "n_t30_eligible_exact_sequences": len(eligible),
        "n_removed_by_t30_eligibility": n_exact_unique - len(eligible),
        "class_counts": json.dumps({str(k): int(v) for k, v in counts.items()}),
    })

task_audit = pd.DataFrame(task_audit_rows)
label_mapping = pd.DataFrame(label_mapping_rows).drop_duplicates()

task_audit.to_csv(
    RQ29A_OUT / "rq29a_task_evidence_preflight.tsv", sep="\t", index=False
)
label_mapping.to_csv(
    RQ29A_OUT / "rq29a_label_mapping.tsv", sep="\t", index=False
)

display(task_audit)
display(label_mapping)

print("Union T30-eligible production sequence_uids:", len(union_eligible_uids))
print("STAGE06_PRODUCTION_EVIDENCE_OK")


,task,task_type,stage06_path,n_stage06_rows,n_stage06_exact_unique,n_noncanonical_rows,n_length_gt_1024_rows,n_t30_eligible_exact_sequences,n_removed_by_t30_eligibility,class_counts
0,global_mixed,binary,/home/david/Desktop/colabs/inta_project/phagel...,70905,70905,178,1279,69448,1457,"{""0"": 58189, ""1"": 11259}"
1,multiclass_clean,multiclass,/home/david/Desktop/colabs/inta_project/phagel...,10841,10841,11,586,10244,597,"{""0"": 9371, ""1"": 635, ""2"": 238}"
2,subtype_endolysin,binary,/home/david/Desktop/colabs/inta_project/phagel...,68926,68926,172,1194,67560,1366,"{""0"": 58189, ""1"": 9371}"
3,subtype_depolymerase,binary,/home/david/Desktop/colabs/inta_project/phagel...,59750,59750,168,758,58824,926,"{""0"": 58189, ""1"": 635}"
4,subtype_val,binary,/home/david/Desktop/colabs/inta_project/phagel...,59279,59279,166,686,58427,852,"{""0"": 58189, ""1"": 238}"


,task,raw_label,production_label,canonical_name
0,global_mixed,1,1,positive
1,global_mixed,0,0,negative
2,multiclass_clean,0,0,endolysin
3,multiclass_clean,1,1,depolymerase
4,multiclass_clean,2,2,virion_associated_lytic_enzyme
5,subtype_endolysin,1,1,positive
6,subtype_endolysin,0,0,negative
7,subtype_depolymerase,1,1,positive
8,subtype_depolymerase,0,0,negative
9,subtype_val,1,1,positive


Union T30-eligible production sequence_uids: 69448
STAGE06_PRODUCTION_EVIDENCE_OK


## 6. Load only the shared T30 rows required by production tasks

In [7]:

wanted_uids = sorted(union_eligible_uids)
required_rep_columns = ["sequence_uid", *FEATURE_COLS]

filter_used = False
try:
    rep = pd.read_parquet(
        REP_PATH,
        columns=required_rep_columns,
        filters=[("sequence_uid", "in", wanted_uids)],
    )
    filter_used = True
except Exception as exc:
    print("Parquet predicate filtering unavailable; falling back to selected-column load.")
    print("Filter exception:", type(exc).__name__, str(exc)[:300])
    rep = pd.read_parquet(REP_PATH, columns=required_rep_columns)
    rep = rep[rep["sequence_uid"].astype(str).isin(union_eligible_uids)].copy()

if list(rep.columns) != required_rep_columns:
    # Column order is contractual for the serialized estimators.
    missing = set(required_rep_columns) - set(rep.columns)
    if missing:
        raise KeyError(f"T30 representation missing columns: {sorted(missing)[:20]}")
    rep = rep[required_rep_columns].copy()

rep["sequence_uid"] = rep["sequence_uid"].astype(str)

if rep["sequence_uid"].duplicated().any():
    raise RuntimeError("T30 representation contains duplicate sequence_uid rows.")

observed = set(rep["sequence_uid"])
missing_uids = union_eligible_uids - observed
if missing_uids:
    raise RuntimeError(
        f"READY T30 representation is missing {len(missing_uids)} expected eligible "
        f"production sequence_uids. First missing: {sorted(missing_uids)[:10]}"
    )

extra = observed - union_eligible_uids
if extra:
    rep = rep[rep["sequence_uid"].isin(union_eligible_uids)].copy()

X_check = rep[FEATURE_COLS].to_numpy(dtype=np.float32, copy=False)
if not np.isfinite(X_check).all():
    raise RuntimeError("Non-finite values detected in production T30 feature subset.")

rep = rep.sort_values("sequence_uid").reset_index(drop=True)
rep_index = rep.set_index("sequence_uid", drop=True)

rep_audit = pd.DataFrame([{
    "representation_id": REPRESENTATION_ID,
    "representation_path": str(REP_PATH),
    "n_required_sequence_uids": len(union_eligible_uids),
    "n_loaded_sequence_uids": len(rep),
    "n_missing": 0,
    "n_features": len(FEATURE_COLS),
    "feature_first": FEATURE_COLS[0],
    "feature_last": FEATURE_COLS[-1],
    "all_finite": True,
    "predicate_filter_used": filter_used,
}])
rep_audit.to_csv(
    RQ29A_OUT / "rq29a_representation_preflight.tsv", sep="\t", index=False
)
display(rep_audit)

print("T30_PRODUCTION_MATRIX_PREFLIGHT_OK")


,representation_id,representation_path,n_required_sequence_uids,n_loaded_sequence_uids,n_missing,n_features,feature_first,feature_last,all_finite,predicate_filter_used
0,esm2_t30_150m,/home/david/Desktop/colabs/inta_project/phagel...,69448,69448,0,640,p_0,p_639,True,True


T30_PRODUCTION_MATRIX_PREFLIGHT_OK


## 7. Estimator construction, imbalance handling, and fit helpers

In [8]:

def build_estimator(config_id, task_type, y):
    cfg = CONFIGS[config_id]
    alg = cfg["algorithm"]

    if alg == "LR":
        model = LogisticRegression(
            C=cfg["C"],
            penalty="l2",
            solver="lbfgs",
            max_iter=2000,
            class_weight="balanced" if task_type == "binary" else None,
            random_state=SEED,
        )
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", model),
        ])

    if alg == "KNN":
        model = KNeighborsClassifier(
            n_neighbors=cfg["n_neighbors"],
            weights="distance",
            metric="cosine",
            algorithm="brute",
            n_jobs=N_JOBS,
        )
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", model),
        ])

    if alg == "MLP":
        model = MLPClassifier(
            hidden_layer_sizes=cfg["hidden_layer_sizes"],
            activation="relu",
            solver="adam",
            alpha=cfg["alpha"],
            learning_rate_init=cfg["learning_rate_init"],
            batch_size=256,
            max_iter=300,
            random_state=SEED,
        )
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", model),
        ])

    if alg == "XGB":
        try:
            from xgboost import XGBClassifier
        except ImportError as exc:
            raise RuntimeError("xgboost is required for the frozen RQ28 recipe.") from exc

        common = dict(
            n_estimators=cfg["n_estimators"],
            max_depth=cfg["max_depth"],
            learning_rate=cfg["learning_rate"],
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            n_jobs=N_JOBS,
            random_state=SEED,
        )

        if task_type == "binary":
            positives = max(1, int(np.sum(np.asarray(y) == 1)))
            negatives = max(1, int(np.sum(np.asarray(y) == 0)))
            return XGBClassifier(
                **common,
                objective="binary:logistic",
                eval_metric="logloss",
                scale_pos_weight=negatives / positives,
            )

        return XGBClassifier(
            **common,
            objective="multi:softprob",
            num_class=3,
            eval_metric="mlogloss",
        )

    raise ValueError(f"Unsupported production algorithm: {alg}")


def fit_estimator(estimator, config_id, task_type, X, y):
    alg = CONFIGS[config_id]["algorithm"]
    fit_meta = {
        "imbalance_policy": "none",
        "sample_weight_used": False,
    }

    if alg == "MLP":
        supports_weight = (
            "sample_weight" in inspect.signature(MLPClassifier.fit).parameters
        )
        if supports_weight:
            sw = compute_sample_weight(class_weight="balanced", y=y)
            estimator.fit(X, y, model__sample_weight=sw)
            fit_meta.update({
                "imbalance_policy": "balanced_sample_weight",
                "sample_weight_used": True,
            })
        else:
            estimator.fit(X, y)
            fit_meta.update({
                "imbalance_policy": "balanced_sample_weight_not_supported_by_installed_sklearn",
                "sample_weight_used": False,
            })
        return estimator, fit_meta

    if alg == "XGB" and task_type == "multiclass":
        sw = compute_sample_weight(class_weight="balanced", y=y)
        estimator.fit(X, y, sample_weight=sw)
        fit_meta.update({
            "imbalance_policy": "balanced_sample_weight",
            "sample_weight_used": True,
        })
        return estimator, fit_meta

    estimator.fit(X, y)

    if alg == "LR" and task_type == "binary":
        fit_meta["imbalance_policy"] = "class_weight_balanced"
    elif alg == "XGB" and task_type == "binary":
        fit_meta["imbalance_policy"] = "scale_pos_weight_from_full_production_fit"
    elif alg == "KNN":
        fit_meta["imbalance_policy"] = "none_by_frozen_contract"

    return estimator, fit_meta


def prediction_payload(model, X):
    pred = np.asarray(model.predict(X))
    proba = np.asarray(model.predict_proba(X), dtype=np.float64)
    classes = np.asarray(model.classes_)
    return pred, proba, classes


print("MLP sample_weight supported:", "sample_weight" in inspect.signature(MLPClassifier.fit).parameters)
print("ESTIMATOR_FACTORY_OK")


MLP sample_weight supported: True
ESTIMATOR_FACTORY_OK



## 8. Fit all frozen members on complete eligible evidence and verify serialization

No performance claim is calculated in this section. The probe rows are used only to verify the serialization invariant.


In [9]:

consensus_lookup = {
    (r.task, r.algorithm): r.consensus_config_id
    for r in config_consensus.itertuples(index=False)
}

roundtrip_rows = []
fit_rows = []
task_release_rows = []

for task in PRODUCTION_TASKS:
    spec = TASK_SPECS[task]
    tf = task_frames[task].sort_values("sequence_uid").reset_index(drop=True)

    if not set(tf["sequence_uid"]).issubset(rep_index.index):
        raise RuntimeError(f"{task}: representation coverage changed after preflight.")

    X = (
        rep_index.loc[tf["sequence_uid"], FEATURE_COLS]
        .to_numpy(dtype=np.float32, copy=True)
    )
    y = tf["y"].to_numpy(dtype=np.int64)

    if len(X) != len(y) or X.shape[1] != 640:
        raise RuntimeError(f"{task}: invalid final fit matrix shape {X.shape}")

    task_dir = STAGING_ROOT / "tasks" / task
    members_dir = task_dir / "members"
    members_dir.mkdir(parents=True, exist_ok=True)

    component = frozen[frozen["task"].eq(task)].iloc[0].to_dict()
    task_members = (
        members[members["task"].eq(task)]
        .sort_values("member_index")
        .reset_index(drop=True)
    )

    fitted_member_records = []

    probe_idx = np.arange(min(PROBE_N, len(y)), dtype=int)
    X_probe = X[probe_idx]

    for rec in task_members.to_dict("records"):
        algorithm = rec["algorithm"]
        config_id = consensus_lookup[(task, algorithm)]
        estimator = build_estimator(config_id, spec["type"], y)

        start = time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            estimator, fit_meta = fit_estimator(
                estimator, config_id, spec["type"], X, y
            )
        fit_seconds = time.perf_counter() - start

        pred_before, proba_before, classes_before = prediction_payload(
            estimator, X_probe
        )

        model_filename = f"{algorithm.lower()}.joblib"
        model_path = members_dir / model_filename
        joblib.dump(estimator, model_path, compress=3)

        reloaded = joblib.load(model_path)
        pred_after, proba_after, classes_after = prediction_payload(
            reloaded, X_probe
        )

        same_pred = bool(np.array_equal(pred_before, pred_after))
        same_classes = bool(np.array_equal(classes_before, classes_after))
        max_abs_proba_diff = float(
            np.max(np.abs(proba_before - proba_after))
            if proba_before.size
            else 0.0
        )
        roundtrip_ok = (
            same_pred
            and same_classes
            and max_abs_proba_diff <= ROUNDTRIP_ATOL
        )

        if not roundtrip_ok:
            raise RuntimeError(
                f"Serialization round-trip failed for {task}/{algorithm}: "
                f"same_pred={same_pred}, same_classes={same_classes}, "
                f"max_abs_proba_diff={max_abs_proba_diff}"
            )

        warning_messages = [str(w.message) for w in caught]

        member_metadata = {
            "schema_version": "rq29a-member-v1",
            "model_version": MODEL_VERSION,
            "task": task,
            "component_id": component["component_id"],
            "task_type": spec["type"],
            "strategy_id": rec["strategy_id"],
            "representation_id": REPRESENTATION_ID,
            "algorithm": algorithm,
            "config_id": config_id,
            "config_parameters": {
                k: v for k, v in CONFIGS[config_id].items()
                if k != "algorithm"
            },
            "random_state": SEED,
            "n_features": 640,
            "feature_columns": {
                "first": FEATURE_COLS[0],
                "last": FEATURE_COLS[-1],
                "ordered_count": len(FEATURE_COLS),
            },
            "fit_rows": int(len(y)),
            "class_counts": {
                str(k): int(v)
                for k, v in pd.Series(y).value_counts().sort_index().items()
            },
            "classes_": [str(x) for x in classes_before.tolist()],
            "preprocessing": (
                "StandardScaler fitted on complete production evidence"
                if algorithm in {"LR", "KNN", "MLP"}
                else "raw FP32 ESM2-T30 embeddings"
            ),
            "imbalance_policy": fit_meta["imbalance_policy"],
            "sample_weight_used": bool(fit_meta["sample_weight_used"]),
            "fit_seconds": float(fit_seconds),
            "fit_warning_count": len(warning_messages),
            "fit_warnings": warning_messages,
            "serialization": {
                "format": "joblib",
                "roundtrip_probe_n": int(len(probe_idx)),
                "probability_atol": ROUNDTRIP_ATOL,
                "roundtrip_ok": True,
            },
        }

        metadata_path = members_dir / f"{algorithm.lower()}.metadata.json"
        metadata_path.write_text(
            json.dumps(member_metadata, indent=2, default=str),
            encoding="utf-8",
        )

        fitted_member_records.append({
            "algorithm": algorithm,
            "strategy_id": rec["strategy_id"],
            "config_id": config_id,
            "model_file": f"members/{model_filename}",
            "metadata_file": f"members/{metadata_path.name}",
        })

        fit_rows.append({
            "task": task,
            "task_type": spec["type"],
            "component_id": component["component_id"],
            "algorithm": algorithm,
            "strategy_id": rec["strategy_id"],
            "config_id": config_id,
            "n_fit": int(len(y)),
            "n_features": int(X.shape[1]),
            "class_counts": json.dumps(
                {str(k): int(v) for k, v in pd.Series(y).value_counts().sort_index().items()}
            ),
            "fit_seconds": float(fit_seconds),
            "imbalance_policy": fit_meta["imbalance_policy"],
            "sample_weight_used": bool(fit_meta["sample_weight_used"]),
            "warning_count": len(warning_messages),
            "model_relpath": f"tasks/{task}/members/{model_filename}",
        })

        roundtrip_rows.append({
            "task": task,
            "algorithm": algorithm,
            "config_id": config_id,
            "probe_n": int(len(probe_idx)),
            "same_predictions": same_pred,
            "same_classes": same_classes,
            "max_abs_probability_difference": max_abs_proba_diff,
            "atol": ROUNDTRIP_ATOL,
            "roundtrip_ok": roundtrip_ok,
        })

        del reloaded, estimator

    label_rows_task = label_mapping[label_mapping["task"].eq(task)]
    production_label_map = {
        str(int(row.production_label)): row.canonical_name
        for row in label_rows_task.itertuples(index=False)
    }

    task_metadata = {
        "schema_version": "rq29a-task-v1",
        "model_version": MODEL_VERSION,
        "task": task,
        "component_id": component["component_id"],
        "tier": component["tier"],
        "default_execution": as_bool(component["default_execution"]),
        "task_type": spec["type"],
        "option_type": component["option_type"],
        "voting_scheme": component["voting_scheme"],
        "n_members": int(component["n_members"]),
        "representation_id": REPRESENTATION_ID,
        "members": fitted_member_records,
        "production_label_map": production_label_map,
        "n_fit_exact_sequences": int(len(y)),
        "source_stage06_dataset": str(
            TASK_ROOT / spec["path"] / "dataset.tsv.gz"
        ),
        "benchmark_context": {
            "mean_mcc": (
                float(component["benchmark_mean_mcc"])
                if "benchmark_mean_mcc" in component and pd.notna(component["benchmark_mean_mcc"])
                else None
            ),
            "worst_mcc": (
                float(component["benchmark_worst_mcc"])
                if "benchmark_worst_mcc" in component and pd.notna(component["benchmark_worst_mcc"])
                else None
            ),
            "strict_R30S30_mcc": (
                float(component["benchmark_strict_R30S30_mcc"])
                if "benchmark_strict_R30S30_mcc" in component and pd.notna(component["benchmark_strict_R30S30_mcc"])
                else None
            ),
            "note": "Selection context from RQ28; full held-out characterization is generated in RQ29B.",
        },
    }

    (task_dir / "metadata.json").write_text(
        json.dumps(task_metadata, indent=2, default=str),
        encoding="utf-8",
    )

    if str(component["option_type"]) == "ensemble":
        member_files = [x["model_file"] for x in fitted_member_records]
        ensemble_payload = {
            "schema_version": "rq29a-ensemble-v1",
            "type": "unweighted_soft_voting",
            "task": task,
            "members": member_files,
            "weights": [1.0 / len(member_files)] * len(member_files),
        }
        if spec["type"] == "binary":
            ensemble_payload.update({
                "threshold": 0.5,
                "positive_class": 1,
            })
        else:
            ensemble_payload.update({
                "class_order": MULTICLASS_ORDER,
                "encoded_class_order": [0, 1, 2],
                "decision": "argmax",
            })

        (task_dir / "ensemble.json").write_text(
            json.dumps(ensemble_payload, indent=2),
            encoding="utf-8",
        )

    task_release_rows.append({
        "task": task,
        "component_id": component["component_id"],
        "tier": component["tier"],
        "default_execution": as_bool(component["default_execution"]),
        "option_type": component["option_type"],
        "n_members": int(component["n_members"]),
        "n_fit": int(len(y)),
        "member_algorithms": ";".join(task_members["algorithm"].tolist()),
    })

    del X, y

fit_audit = pd.DataFrame(fit_rows)
roundtrip_audit = pd.DataFrame(roundtrip_rows)
task_release = pd.DataFrame(task_release_rows)

fit_audit.to_csv(
    RQ29A_OUT / "rq29a_final_fit_audit.tsv", sep="\t", index=False
)
roundtrip_audit.to_csv(
    RQ29A_OUT / "rq29a_roundtrip_audit.tsv", sep="\t", index=False
)
task_release.to_csv(
    RQ29A_OUT / "rq29a_task_release_summary.tsv", sep="\t", index=False
)

display(fit_audit)
display(roundtrip_audit)
display(task_release)

print("ALL_FROZEN_MEMBERS_FIT_AND_ROUNDTRIP_OK")


,task,task_type,component_id,algorithm,strategy_id,config_id,n_fit,n_features,class_counts,fit_seconds,imbalance_policy,sample_weight_used,warning_count,model_relpath
0,global_mixed,binary,core_global_lytic,KNN,esm2_t30_150m + KNN,KNN02,69448,640,"{""0"": 58189, ""1"": 11259}",0.510619,none_by_frozen_contract,False,0,tasks/global_mixed/members/knn.joblib
1,global_mixed,binary,core_global_lytic,MLP,esm2_t30_150m + MLP,MLP02,69448,640,"{""0"": 58189, ""1"": 11259}",220.525716,balanced_sample_weight,True,0,tasks/global_mixed/members/mlp.joblib
2,multiclass_clean,multiclass,core_function_multiclass,XGB,esm2_t30_150m + XGB,XGB01,10244,640,"{""0"": 9371, ""1"": 635, ""2"": 238}",41.111566,balanced_sample_weight,True,0,tasks/multiclass_clean/members/xgb.joblib
3,multiclass_clean,multiclass,core_function_multiclass,LR,esm2_t30_150m + LR,LR01,10244,640,"{""0"": 9371, ""1"": 635, ""2"": 238}",10.309608,none,False,1,tasks/multiclass_clean/members/lr.joblib
4,subtype_endolysin,binary,aux_endolysin_compatibility,KNN,esm2_t30_150m + KNN,KNN01,67560,640,"{""0"": 58189, ""1"": 9371}",0.447692,none_by_frozen_contract,False,0,tasks/subtype_endolysin/members/knn.joblib
5,subtype_endolysin,binary,aux_endolysin_compatibility,XGB,esm2_t30_150m + XGB,XGB01,67560,640,"{""0"": 58189, ""1"": 9371}",37.723490,scale_pos_weight_from_full_production_fit,False,0,tasks/subtype_endolysin/members/xgb.joblib
6,subtype_endolysin,binary,aux_endolysin_compatibility,MLP,esm2_t30_150m + MLP,MLP02,67560,640,"{""0"": 58189, ""1"": 9371}",255.556911,balanced_sample_weight,True,0,tasks/subtype_endolysin/members/mlp.joblib
7,subtype_depolymerase,binary,aux_depolymerase_compatibility,MLP,esm2_t30_150m + MLP,MLP01,58824,640,"{""0"": 58189, ""1"": 635}",66.352087,balanced_sample_weight,True,0,tasks/subtype_depolymerase/members/mlp.joblib
8,subtype_depolymerase,binary,aux_depolymerase_compatibility,KNN,esm2_t30_150m + KNN,KNN03,58824,640,"{""0"": 58189, ""1"": 635}",0.400308,none_by_frozen_contract,False,0,tasks/subtype_depolymerase/members/knn.joblib
9,subtype_val,binary,aux_VAL_compatibility,MLP,esm2_t30_150m + MLP,MLP01,58427,640,"{""0"": 58189, ""1"": 238}",45.976443,balanced_sample_weight,True,0,tasks/subtype_val/members/mlp.joblib


,task,algorithm,config_id,probe_n,same_predictions,same_classes,max_abs_probability_difference,atol,roundtrip_ok
0,global_mixed,KNN,KNN02,8,True,True,0.0,1.000000e-10,True
1,global_mixed,MLP,MLP02,8,True,True,0.0,1.000000e-10,True
2,multiclass_clean,XGB,XGB01,8,True,True,0.0,1.000000e-10,True
3,multiclass_clean,LR,LR01,8,True,True,0.0,1.000000e-10,True
4,subtype_endolysin,KNN,KNN01,8,True,True,0.0,1.000000e-10,True
5,subtype_endolysin,XGB,XGB01,8,True,True,0.0,1.000000e-10,True
6,subtype_endolysin,MLP,MLP02,8,True,True,0.0,1.000000e-10,True
7,subtype_depolymerase,MLP,MLP01,8,True,True,0.0,1.000000e-10,True
8,subtype_depolymerase,KNN,KNN03,8,True,True,0.0,1.000000e-10,True
9,subtype_val,MLP,MLP01,8,True,True,0.0,1.000000e-10,True


,task,component_id,tier,default_execution,option_type,n_members,n_fit,member_algorithms
0,global_mixed,core_global_lytic,core,True,ensemble,2,69448,KNN;MLP
1,multiclass_clean,core_function_multiclass,core,True,ensemble,2,10244,XGB;LR
2,subtype_endolysin,aux_endolysin_compatibility,auxiliary,False,ensemble,3,67560,KNN;XGB;MLP
3,subtype_depolymerase,aux_depolymerase_compatibility,auxiliary,False,ensemble,2,58824,MLP;KNN
4,subtype_val,aux_VAL_compatibility,auxiliary,False,single,1,58427,MLP


ALL_FROZEN_MEMBERS_FIT_AND_ROUNDTRIP_OK


## 9. Representation provenance and release metadata

In [10]:

def sha256_file(path, block_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            h.update(block)
    return h.hexdigest()


representation_dir = STAGING_ROOT / "representation"
representation_dir.mkdir(parents=True, exist_ok=True)

print("Hashing frozen T30 representation artifact for release provenance...")
rep_sha256 = sha256_file(REP_PATH)

representation_metadata = {
    "schema_version": "rq29a-representation-v1",
    "representation_id": REPRESENTATION_ID,
    "checkpoint": REPRESENTATION_CHECKPOINT,
    "modality": "protein_sequence_embedding",
    "pooling": "mean_sequence_pooling",
    "hidden_layer": "final",
    "dtype": "FP32",
    "dimension": 640,
    "feature_columns": {
        "first": FEATURE_COLS[0],
        "last": FEATURE_COLS[-1],
        "ordered_count": len(FEATURE_COLS),
    },
    "eligibility": {
        "canonical_20aa_only": True,
        "max_sequence_length": 1024,
    },
    "source_artifact": str(REP_PATH),
    "source_sha256": rep_sha256,
    "n_production_sequence_uids_loaded": int(len(rep)),
    "shared_across_all_frozen_components": True,
}

representation_metadata_path = (
    representation_dir / f"{REPRESENTATION_ID}.json"
)
representation_metadata_path.write_text(
    json.dumps(representation_metadata, indent=2),
    encoding="utf-8",
)

print("REPRESENTATION_PROVENANCE_WRITTEN")


Hashing frozen T30 representation artifact for release provenance...
REPRESENTATION_PROVENANCE_WRITTEN


## 10. Build checksums, release manifest, and atomically publish the production bundle

In [11]:

# Source artifact hashes connect production weights back to the frozen architecture
# and the complete Stage-06 evidence tables.
source_artifacts = {
    "rq28_frozen_components": {
        "path": str(RQ28_COMPONENTS_PATH),
        "sha256": sha256_file(RQ28_COMPONENTS_PATH),
    },
    "rq28_production_architecture": {
        "path": str(RQ28_ARCH_PATH),
        "sha256": sha256_file(RQ28_ARCH_PATH),
    },
    "representation_registry": {
        "path": str(REGISTRY_PATH),
        "sha256": sha256_file(REGISTRY_PATH),
    },
    "representation_artifact": {
        "path": str(REP_PATH),
        "sha256": rep_sha256,
    },
}

for task in PRODUCTION_TASKS:
    p = TASK_ROOT / TASK_SPECS[task]["path"] / "dataset.tsv.gz"
    source_artifacts[f"stage06::{task}"] = {
        "path": str(p),
        "sha256": sha256_file(p),
    }

# First checksum all model/task/representation payloads (excluding manifest/checksums).
inventory_rows = []
for p in sorted(STAGING_ROOT.rglob("*")):
    if not p.is_file():
        continue
    rel = p.relative_to(STAGING_ROOT).as_posix()
    if rel in {"manifest.json", "checksums.tsv"}:
        continue
    inventory_rows.append({
        "relative_path": rel,
        "bytes": int(p.stat().st_size),
        "sha256": sha256_file(p),
    })

checksums = pd.DataFrame(inventory_rows)
checksums.to_csv(
    STAGING_ROOT / "checksums.tsv", sep="\t", index=False
)

release_manifest = {
    "schema_version": "phagelysclass-model-release-v1",
    "model_version": MODEL_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "architecture_id": architecture["architecture_id"],
    "architecture_label": architecture.get("architecture_label"),
    "production_global_task": architecture["production_global_task"],
    "random_state": SEED,
    "representation_id": REPRESENTATION_ID,
    "representation_checkpoint": REPRESENTATION_CHECKPOINT,
    "n_components": 5,
    "n_core_components": 2,
    "n_auxiliary_components": 3,
    "n_fitted_estimators": int(len(fit_audit)),
    "default_execution_estimators": int(
        fit_audit["task"].isin(
            frozen.loc[frozen["default_execution"].map(as_bool), "task"]
        ).sum()
    ),
    "configuration_consolidation": {
        "source": "validation-only selections from 16 R/S benchmark cells",
        "primary": "modal selected config_id",
        "tie_breaks": [
            "median validation MCC",
            "median validation AP for binary or macro-F1 for multiclass",
            "config_order",
            "config_id",
        ],
        "outer_test_used": False,
    },
    "final_refit": {
        "source": "complete Stage-06 task dataset after frozen T30 eligibility",
        "redundancy_regime": None,
        "homology_split_regime": None,
        "reason": (
            "R and S are benchmark perturbation axes and are not deployment hyperparameters."
        ),
    },
    "components": task_release.to_dict("records"),
    "source_artifacts": source_artifacts,
    "checksum_table": "checksums.tsv",
    "n_payload_files_checksummed": int(len(checksums)),
    "software": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": importlib.metadata.version("scikit-learn"),
        "joblib": importlib.metadata.version("joblib"),
        "xgboost": (
            importlib.metadata.version("xgboost")
            if importlib.util.find_spec("xgboost") is not None
            else None
        ),
    },
    "next_stage": (
        "RQ29B performs held-out characterization of frozen individual members "
        "and final ensemble recipes. Training-fit performance is intentionally absent here."
    ),
}

(STAGING_ROOT / "manifest.json").write_text(
    json.dumps(release_manifest, indent=2, default=str),
    encoding="utf-8",
)

# External inventory includes manifest and checksums too.
external_inventory_rows = []
for p in sorted(STAGING_ROOT.rglob("*")):
    if p.is_file():
        external_inventory_rows.append({
            "relative_path": p.relative_to(STAGING_ROOT).as_posix(),
            "bytes": int(p.stat().st_size),
            "sha256": sha256_file(p),
        })
external_inventory = pd.DataFrame(external_inventory_rows)
external_inventory.to_csv(
    RQ29A_OUT / "rq29a_export_inventory.tsv", sep="\t", index=False
)

rq29a_manifest = {
    "stage": "RQ29A",
    "status": "built_in_staging",
    "production_model_root": str(FINAL_MODEL_ROOT),
    "n_members": int(len(fit_audit)),
    "n_tasks": 5,
    "roundtrip_all_ok": bool(roundtrip_audit["roundtrip_ok"].all()),
    "config_selection_cells": int(len(selected_cells)),
    "representation_id": REPRESENTATION_ID,
    "rq28_architecture_id": architecture["architecture_id"],
    "audit_exports": [
        "rq29a_selected_config_cells.tsv",
        "rq29a_config_consensus.tsv",
        "rq29a_task_evidence_preflight.tsv",
        "rq29a_label_mapping.tsv",
        "rq29a_representation_preflight.tsv",
        "rq29a_final_fit_audit.tsv",
        "rq29a_roundtrip_audit.tsv",
        "rq29a_task_release_summary.tsv",
        "rq29a_export_inventory.tsv",
        "rq29a_manifest.json",
    ],
}
(RQ29A_OUT / "rq29a_manifest.json").write_text(
    json.dumps(rq29a_manifest, indent=2),
    encoding="utf-8",
)

# Atomic-ish publication: preserve an existing release until the new staging tree is complete.
backup = MODELS_PARENT / f".{MODEL_VERSION}.backup"
if backup.exists():
    shutil.rmtree(backup)

if FINAL_MODEL_ROOT.exists():
    if not ALLOW_REBUILD:
        raise FileExistsError(FINAL_MODEL_ROOT)
    FINAL_MODEL_ROOT.rename(backup)

try:
    STAGING_ROOT.rename(FINAL_MODEL_ROOT)
except Exception:
    if backup.exists() and not FINAL_MODEL_ROOT.exists():
        backup.rename(FINAL_MODEL_ROOT)
    raise
else:
    if backup.exists():
        shutil.rmtree(backup)

rq29a_manifest["status"] = "published"
(RQ29A_OUT / "rq29a_manifest.json").write_text(
    json.dumps(rq29a_manifest, indent=2),
    encoding="utf-8",
)

print("Published:", FINAL_MODEL_ROOT)
print("PRODUCTION_BUNDLE_PUBLISHED")


Published: /home/david/Desktop/colabs/inta_project/phagelytic_core_project/phagelysclass/models/phagelysclass_v1
PRODUCTION_BUNDLE_PUBLISHED


## 11. RQ29A execution gates

In [12]:

required_audit_exports = [
    "rq29a_selected_config_cells.tsv",
    "rq29a_config_consensus.tsv",
    "rq29a_task_evidence_preflight.tsv",
    "rq29a_label_mapping.tsv",
    "rq29a_representation_preflight.tsv",
    "rq29a_final_fit_audit.tsv",
    "rq29a_roundtrip_audit.tsv",
    "rq29a_task_release_summary.tsv",
    "rq29a_export_inventory.tsv",
    "rq29a_manifest.json",
]

assert len(master) == 2240
assert architecture["architecture_id"] == "C_2_core_plus_3_aux"
assert architecture["production_global_task"] == "global_mixed"
assert len(frozen) == 5
assert len(members) == 10
assert set(members["representation"]) == {REPRESENTATION_ID}

assert len(selected_cells) == 160
assert selected_cells["selection_match"].all()
assert len(config_consensus) == 10
assert config_consensus["consensus_config_id"].notna().all()

assert len(task_audit) == 5
assert len(rep) == len(union_eligible_uids)
assert rep.shape[1] == 641
assert len(FEATURE_COLS) == 640

assert len(fit_audit) == 10
assert len(roundtrip_audit) == 10
assert roundtrip_audit["roundtrip_ok"].all()
assert roundtrip_audit["max_abs_probability_difference"].le(
    ROUNDTRIP_ATOL
).all()

assert FINAL_MODEL_ROOT.is_dir()
assert not STAGING_ROOT.exists()
assert (FINAL_MODEL_ROOT / "manifest.json").is_file()
assert (FINAL_MODEL_ROOT / "checksums.tsv").is_file()
assert (
    FINAL_MODEL_ROOT / "representation" / f"{REPRESENTATION_ID}.json"
).is_file()

joblib_files = sorted(FINAL_MODEL_ROOT.glob("tasks/*/members/*.joblib"))
task_metadata_files = sorted(FINAL_MODEL_ROOT.glob("tasks/*/metadata.json"))
ensemble_files = sorted(FINAL_MODEL_ROOT.glob("tasks/*/ensemble.json"))

assert len(joblib_files) == 10
assert len(task_metadata_files) == 5
assert len(ensemble_files) == 4  # all except the frozen single-model VAL auxiliary

# Recheck every payload checksum after publication.
published_checksums = pd.read_csv(
    FINAL_MODEL_ROOT / "checksums.tsv", sep="\t", low_memory=False
)
checksum_failures = []
for row in published_checksums.itertuples(index=False):
    p = FINAL_MODEL_ROOT / row.relative_path
    if not p.is_file() or sha256_file(p) != row.sha256:
        checksum_failures.append(row.relative_path)
assert not checksum_failures, checksum_failures

assert all((RQ29A_OUT / name).is_file() for name in required_audit_exports)

print("RQ29A_EXECUTION_GATES_OK")
print("Architecture:", architecture["architecture_id"])
print("Production tasks:", len(task_metadata_files))
print("Serialized fitted estimators:", len(joblib_files))
print("Explicit soft-voting recipes:", len(ensemble_files))
print("Shared representation:", REPRESENTATION_ID)
print("Validation-only config-selection cells audited:", len(selected_cells))
print("Serialization round-trips:", int(roundtrip_audit["roundtrip_ok"].sum()), "/", len(roundtrip_audit))
print("Production bundle:", FINAL_MODEL_ROOT)
print()
print(
    "Interpretation status: RQ29A execution gates passed in this runtime. "
    "Assign REAL-DATA PASS only after the executed real-project notebook is inspected "
    "for complete execution, zero errors/stderr, and coherent exported artifacts."
)


RQ29A_EXECUTION_GATES_OK
Architecture: C_2_core_plus_3_aux
Production tasks: 5
Serialized fitted estimators: 10
Explicit soft-voting recipes: 4
Shared representation: esm2_t30_150m
Validation-only config-selection cells audited: 160
Serialization round-trips: 10 / 10
Production bundle: /home/david/Desktop/colabs/inta_project/phagelytic_core_project/phagelysclass/models/phagelysclass_v1

Interpretation status: RQ29A execution gates passed in this runtime. Assign REAL-DATA PASS only after the executed real-project notebook is inspected for complete execution, zero errors/stderr, and coherent exported artifacts.



## RQ29A interpretation guardrails

- These serialized estimators are **production refits**, not new independent benchmark evaluations.
- Do not report training-set performance from the all-data refits.
- RQ29B must characterize the frozen recipes using held-out benchmark predictions.
- `global_mixed` is functional **lytic-vs-background recognition**, not taxonomic phage-vs-nonphage classification.
- Binary subtype models remain auxiliary compatibility signals; they do not override the clean multiclass core.
- VAL remains evidence-limited and must retain its support-aware warning in the later inference layer.
